# Processes Snakebite CSV with Nepal administrative hierarchy data.

Steps:
  1. Extracts "Month Year" from verbose column names and sums duplicate periods.
  2. Splits rows into three DataFrames by org unit level:
       - Province  (organisationunitcode length == 1)
       - District  (organisationunitcode length == 3)
       - Municipality (organisationunitcode length == 5)
  3. Enriches District rows with their Province name/code.
  4. Enriches Municipality rows with their District and Province name/code.
  5. Saves each level as a separate CSV.

Outputs:
    provinces.csv
    districts.csv
    municipalities.csv

In [2]:
# import argparse
import re
import sys
import os
from pathlib import Path

import pandas as pd

In [4]:
import datetime
import nepali_datetime

In [3]:
NEPALI_MONTHS = ['Baishak',
    'Jestha',
    'Asar',
    'Shrawan',
    'Bhadra',
    'Ashwin',
    'Kartik',
    'Mangsir',
    'Poush',
    'Magh',
    'Falgun',
    'Chaitra']

MONTH_PATTERN = re.compile(
    r"(" + "|".join(re.escape(m) for m in NEPALI_MONTHS) + r")\s+(\d{4})",
    re.IGNORECASE,
)

In [4]:
def extract_period(col_name: str) -> str | None:
    """Return 'Month YYYY' if found in col_name, else None."""
    match = MONTH_PATTERN.search(col_name)
    if match:
        month = match.group(1).capitalize()
        year = match.group(2)
        return f"{month} {year}"
    return None


def collapse_period_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename data columns to their extracted period and sum columns that
    share the same period label.  Non-data columns are kept as-is.
    """
    id_cols = ["organisationunitid", "organisationunitname", "organisationunitcode"]
    rename_map: dict[str, str] = {}
    no_period: list[str] = []

    for col in df.columns:
        if col in id_cols:
            continue
        period = extract_period(col)
        if period:
            rename_map[col] = period
        else:
            no_period.append(col)

    if no_period:
        print(
            f"[warning] Could not extract a period from {len(no_period)} column(s); "
            f"they will be dropped:\n  " + "\n  ".join(no_period)
        )

    # Keep only id cols + mappable cols
    keep_cols = id_cols + list(rename_map.keys())
    df = df[keep_cols].rename(columns=rename_map)

    result = df.T.reset_index().groupby("index").sum().T.reset_index(drop=True)
    
    remaining_items = [item for item in list(result.columns) if item not in set(id_cols)]
    new_list = id_cols + remaining_items
    result = result[new_list]

    return result


def code_level(code) -> int | None:
    """Return hierarchy level based on code string length."""
    s = str(code).strip()
    if len(s) == 1:
        return 0  # Province
    if len(s) == 3:
        return 1  # District
    if len(s) == 5:
        return 2  # Municipality
    return None  # unknown


def split_by_level(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split DataFrame into province, district, municipality subsets."""
    levels = df["organisationunitcode"].astype(str).str.strip().str.len().map(
        {1: "province", 3: "district", 5: "municipality"}
    )
    provinces = df[levels == "province"].copy()
    districts = df[levels == "district"].copy()
    municipalities = df[levels == "municipality"].copy()
    return provinces, districts, municipalities


def build_province_lookup(provinces: pd.DataFrame) -> dict[str, tuple[str, str]]:
    """Map province_code_prefix -> (province_name, province_code)."""
    lookup = {}
    for _, row in provinces.iterrows():
        code = str(row["organisationunitcode"]).strip()
        lookup[code] = (row["organisationunitname"], code)
    return lookup


def build_district_lookup(districts: pd.DataFrame) -> dict[str, tuple[str, str]]:
    """Map district_code -> (district_name, district_code)."""
    lookup = {}
    for _, row in districts.iterrows():
        code = str(row["organisationunitcode"]).strip()
        lookup[code] = (row["organisationunitname"], code)
    return lookup


def enrich_districts(
    districts: pd.DataFrame,
    province_lookup: dict,
) -> pd.DataFrame:
    """Add province_name and province_code columns to district rows."""
    prov_names, prov_codes = [], []
    for code in districts["organisationunitcode"].astype(str).str.strip():
        # Province code is first digit of district code
        prov_key = code[0]
        if prov_key in province_lookup:
            name, pcode = province_lookup[prov_key]
        else:
            name, pcode = None, None
        prov_names.append(name)
        prov_codes.append(pcode)

    out = districts.copy()
    out.insert(2, "province_name", prov_names)
    out.insert(3, "province_code", prov_codes)
    return out


def enrich_municipalities(
    municipalities: pd.DataFrame,
    province_lookup: dict,
    district_lookup: dict,
) -> pd.DataFrame:
    """Add district_name, district_code, province_name, province_code to municipality rows."""
    dist_names, dist_codes = [], []
    prov_names, prov_codes = [], []

    for code in municipalities["organisationunitcode"].astype(str).str.strip():
        # District code = first 3 digits; Province code = first digit
        dist_key = code[:3]
        prov_key = code[0]

        if dist_key in district_lookup:
            dname, dcode = district_lookup[dist_key]
        else:
            dname, dcode = None, None

        if prov_key in province_lookup:
            pname, pcode = province_lookup[prov_key]
        else:
            pname, pcode = None, None

        dist_names.append(dname)
        dist_codes.append(dcode)
        prov_names.append(pname)
        prov_codes.append(pcode)

    out = municipalities.copy()
    out.insert(2, "district_name", dist_names)
    out.insert(3, "district_code", dist_codes)
    out.insert(4, "province_name", prov_names)
    out.insert(5, "province_code", prov_codes)
    return out

def convert_dates(df: pd.DataFrame) -> pd.DataFrame:
    """Rename columns in df based on Bikram Sambat dates to datetime conversion"""
    for period in df.columns:
        if period.split()[0] in NEPALI_MONTHS:
            m = NEPALI_MONTHS.index(period.split()[0]) + 1
            y = int(period.split()[1])
            ad_date = nepali_datetime.date(y, m, 12).to_datetime_date()
            df.rename(columns={period: ad_date}, inplace=True)

    return df

In [5]:
df = pd.read_csv('raw_data/nonpoison.csv', dtype={"organisationunitcode": str})
df.pop('Unnamed: 3')

0     NaN
1     NaN
2     NaN
3     NaN
4     NaN
       ..
699   NaN
700   NaN
701   NaN
702   NaN
703   NaN
Name: Unnamed: 3, Length: 704, dtype: float64

In [6]:
df.head(2)

,organisationunitid,organisationunitname,organisationunitcode,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Bhadra 2077,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Chaitra 2077,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Kartik 2074,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Magh 2079,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Asar 2079,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Chaitra 2072,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Mangsir 2073,...,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Shrawan 2072,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Poush 2074,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Kartik 2081,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Shrawan 2075,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Falgun 2074,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Shrawan 2073,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Jestha 2074,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Falgun 2072,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Kartik 2079,OPD-Morbidity-Other Diseases & Injuries-Snake Bite-Non Poisonous Baishak 2074
0,YubZ3SsXpMk,10905 Falgunanda Rural Municipality,10905,4.0,NaN,2.0,NaN,NaN,1.0,NaN,...,1.0,1.0,NaN,3.0,NaN,2.0,1.0,NaN,1.0,NaN
1,UYA23UmgomU,30802 Lalitpur Metropolitan City,30802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN


In [7]:
print("Splitting by administrative level...")
provinces_raw, districts_raw, municipalities_raw = split_by_level(df)
print(f"  Provinces:     {len(provinces_raw):>6,} rows")
print(f"  Districts:     {len(districts_raw):>6,} rows")
print(f"  Municipalities:{len(municipalities_raw):>6,} rows")

unclassified = len(df) - len(provinces_raw) - len(districts_raw) - len(municipalities_raw)
if unclassified:
    print(f"  [warning] {unclassified} rows could not be classified and were dropped.")

Splitting by administrative level...
  Provinces:          7 rows
  Districts:         77 rows
  Municipalities:   620 rows


In [8]:
# ---- Collapse period columns per level independently --------------------
print("Extracting and collapsing period columns...")
provinces_raw   = collapse_period_columns(provinces_raw)
districts_raw   = collapse_period_columns(districts_raw)
municipalities_raw = collapse_period_columns(municipalities_raw)

period_cols = [c for c in provinces_raw.columns if c not in ["organisationunitid", "organisationunitname", "organisationunitcode"]]
print(f"  {len(period_cols)} unique period(s) found: {', '.join(period_cols[:5])}" + "...")

Extracting and collapsing period columns...
  128 unique period(s) found: Asar 2072, Asar 2073, Asar 2074, Asar 2075, Asar 2076...


In [9]:
provinces_raw

index,organisationunitid,organisationunitname,organisationunitcode,Asar 2072,Asar 2073,Asar 2074,Asar 2075,Asar 2076,Asar 2077,Asar 2078,...,Shrawan 2073,Shrawan 2074,Shrawan 2075,Shrawan 2076,Shrawan 2077,Shrawan 2078,Shrawan 2079,Shrawan 2080,Shrawan 2081,Shrawan 2082
0,fvN7GZvNAOB,6 Karnali Province,6,17.0,27.0,9.0,18.0,29.0,23.0,19.0,...,32.0,45.0,22.0,32.0,37.0,38.0,49.0,69.0,39.0,189.0
1,RVc3XoVoNRf,1 Koshi Province,1,70.0,87.0,101.0,270.0,146.0,82.0,1214.0,...,142.0,129.0,137.0,74.0,243.0,240.0,375.0,306.0,532.0,1405.0
2,wtU6v09Kbe0,7 Sudurpashchim Province,7,31.0,27.0,15.0,25.0,27.0,20.0,23.0,...,60.0,34.0,40.0,33.0,26.0,34.0,15.0,37.0,69.0,249.0
3,a6W190BanBu,2 Madhesh Province,2,95.0,176.0,263.0,168.0,54.0,141.0,86.0,...,209.0,196.0,136.0,131.0,88.0,124.0,44.0,173.0,178.0,305.0
4,hi16ZuHEWaY,3 Bagmati Province,3,140.0,322.0,62.0,107.0,95.0,129.0,108.0,...,84.0,104.0,86.0,115.0,137.0,109.0,145.0,128.0,184.0,608.0
5,Zx3boDXh1Q5,5 Lumbini Province,5,168.0,554.0,135.0,188.0,74.0,70.0,300.0,...,223.0,140.0,153.0,92.0,350.0,388.0,510.0,323.0,344.0,1206.0
6,GvgqqErqwFP,4 Gandaki Province,4,107.0,238.0,123.0,169.0,174.0,149.0,185.0,...,249.0,105.0,203.0,244.0,192.0,247.0,224.0,228.0,220.0,809.0


In [10]:
# ---- Convert dates to AD --------------------
print("Converting dates...")
provinces   = convert_dates(provinces_raw)
districts   = convert_dates(districts_raw)
municipalities = convert_dates(municipalities_raw)

Converting dates...


In [11]:
province_lookup = build_province_lookup(provinces)
district_lookup = build_district_lookup(districts)

districts_enriched = enrich_districts(districts, province_lookup)
municipalities_enriched = enrich_municipalities(municipalities, province_lookup, district_lookup)

In [13]:
provinces.head(3)

index,organisationunitid,organisationunitname,organisationunitcode,2015-06-27,2016-06-26,2017-06-26,2018-06-26,2019-06-27,2020-06-26,2021-06-26,...,2016-07-27,2017-07-27,2018-07-28,2019-07-28,2020-07-27,2021-07-27,2022-07-28,2023-07-28,2024-07-27,2025-07-28
0,fvN7GZvNAOB,6 Karnali Province,6,17.0,27.0,9.0,18.0,29.0,23.0,19.0,...,32.0,45.0,22.0,32.0,37.0,38.0,49.0,69.0,39.0,189.0
1,RVc3XoVoNRf,1 Koshi Province,1,70.0,87.0,101.0,270.0,146.0,82.0,1214.0,...,142.0,129.0,137.0,74.0,243.0,240.0,375.0,306.0,532.0,1405.0
2,wtU6v09Kbe0,7 Sudurpashchim Province,7,31.0,27.0,15.0,25.0,27.0,20.0,23.0,...,60.0,34.0,40.0,33.0,26.0,34.0,15.0,37.0,69.0,249.0


In [17]:
proc_data_folder = "proc_data"
out_province = "np_provinces.csv"
out_district = "np_districts.csv"
out_municipality = "np_municipalities.csv"

provinces.to_csv(os.path.join(proc_data_folder, out_province), index=False)
districts_enriched.to_csv(os.path.join(proc_data_folder, out_district), index=False)
municipalities_enriched.to_csv(os.path.join(proc_data_folder, out_municipality), index=False)

print(f"\nSaved:")
print(f"  {out_province}")
print(f"  {out_district}")
print(f"  {out_municipality}")
print("Done.")


Saved:
  np_provinces.csv
  np_districts.csv
  np_municipalities.csv
Done.
